In [3]:
# ==============================================================================
# PANTHORON TRACEAUDIT AGENT - POOF OF CONCEPT (PoC)
# Built for Google AI Hackathon
# ==============================================================================

# Install the necessary Google GenAI SDK (Run this cell first if not installed)
# !pip install -q -U google-genai

import os
from datetime import datetime, timedelta
from google import genai
from google.genai import types
from IPython.display import display, Markdown

# ------------------------------------------------------------------------------
# 1. AUTHENTICATION
# ------------------------------------------------------------------------------
# TODO: Replace the string below with your actual Google Gemini API Key
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"

# Initialize the GenAI Client
client = genai.Client(api_key=GOOGLE_API_KEY)

# ------------------------------------------------------------------------------
# 2. DEFINING THE TOOLS (FUNCTION CALLING)
# ------------------------------------------------------------------------------

def fetch_lot_production_data(lot_number: str) -> str:
    """
    Queries the ERP database (Google Sheets mock) to find exactly where and when
    a contaminated raw material lot was used in the factory.
    """
    print(f"🔧 [TOOL EXECUTION] Querying ERP for Raw Material Lot: {lot_number}...")
    return "Lot found. Used on Production Line 1. Drop time: 03:33:53. Final product code: 09118. Line 1 constraints: 5-minute conveyor time, 35-minute freezing tunnel time."

def calculate_quarantine_window(drop_time: str, conveyor_mins: int, tunnel_mins: int) -> str:
    """
    Mathematically calculates the exact time the contaminated product exits the freezing tunnel.
    """
    print(f"🔧 [TOOL EXECUTION] Calculating quarantine time-shift for drop time: {drop_time}...")
    time_format = "%H:%M:%S"
    drop_dt = datetime.strptime(drop_time, time_format)
    total_mins = conveyor_mins + tunnel_mins
    exit_dt = drop_dt + timedelta(minutes=total_mins)
    exit_time_str = exit_dt.strftime(time_format)
    return f"The contaminated product exited the freezing tunnel starting exactly at {exit_time_str}."

def search_boxes_in_google_sheets(exit_window_start: str) -> str:
    """
    Queries the industrial ERP database to identify the affected Master Pallet LPN
    based on the exit time.
    """
    print(f"🔧 [TOOL EXECUTION] Querying mocked ERP database for production starting at: {exit_window_start}...")
    return "Found matching production batch. Affected Pallet ID is LPN-260724-7153. Pallet status successfully changed to 'BLOCKED' in the database."

def scan_google_drive_for_shipping(pallet_lpn: str) -> str:
    """
    Simulates scanning Google Drive PDFs (Traceability forms / OCR)
    to check if the blocked pallet has already been shipped to a customer.
    """
    print(f"🔧 [TOOL EXECUTION] Scanning simulated Google Drive for shipping documents related to: {pallet_lpn}...")
    return "CRITICAL: Pallet LPN-260724-7153 has been shipped. Document matched: '24072026FINAL.pdf'. Customer: M. OGKOUNSOTO M.IKE, Address: Tsimiski 82, Thessaloniki. Loading Vehicle: NBX7849."

# ------------------------------------------------------------------------------
# 3. AGENT CONFIGURATION (PERSONA & RULES)
# ------------------------------------------------------------------------------
current_date = datetime.now().strftime("%B %d, %Y")

agent_persona = f"""
You are the 'Panthoron TraceAudit Agent', an autonomous Senior Quality Manager for an industrial food factory.
Your primary directive is to handle food safety crises swiftly and accurately.
You strictly follow IFS, BRC, and ISO food safety standards.

When you receive a crisis alert containing a contaminated Lot Number:
1. NEVER guess or hallucinate data.
2. ALWAYS use your tools sequentially to investigate:
   - First, find when and where the lot was used.
   - Second, calculate the physical time constraints (freezing tunnel exit time).
   - Third, find the affected Pallet LPN.
   - Fourth, check logistics to see if it has been shipped.
3. Synthesize all data into a highly professional 'OFFICIAL URGENT RECALL REPORT'.
4. Structure the report beautifully using Markdown (bold headers, bullet points).
5. CRITICAL RULE: The exact current date is {current_date}. You MUST use this exact date in the DATE field of your official report.
"""

# ------------------------------------------------------------------------------
# 4. THE CRISIS SCENARIO (USER PROMPT)
# ------------------------------------------------------------------------------
# The Agent receives ONLY the raw email from the supplier. It must figure out the rest!

crisis_email = """
URGENT NOTIFICATION FROM SUPPLIER:
We just detected severe Escherichia coli (E. coli) contamination in Raw Material Lot: 260707AH.
Please investigate immediately.
"""

# ------------------------------------------------------------------------------
# 5. EXECUTING THE AGENTIC WORKFLOW
# ------------------------------------------------------------------------------

print("🤖 [AGENT] Analyzing crisis prompt and initializing trace workflow...\n")

# Create a Chat Session using Gemini 3.5 Flash and enable our tools
chat = client.chats.create(
    model="gemini-3.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=agent_persona,
        # Notice we now have 4 tools!
        tools=[fetch_lot_production_data, calculate_quarantine_window, search_boxes_in_google_sheets, scan_google_drive_for_shipping],
        temperature=0.1,
    )
)

response = chat.send_message(crisis_email)

# ------------------------------------------------------------------------------
# 6. DISPLAYING THE FINAL RECALL REPORT
# ------------------------------------------------------------------------------

print("\n" + "="*80)
display(Markdown(response.text))
print("="*80)

🤖 [AGENT] Analyzing crisis prompt and initializing trace workflow...

🔧 [TOOL EXECUTION] Querying ERP for Raw Material Lot: 260707AH...
🔧 [TOOL EXECUTION] Calculating quarantine time-shift for drop time: 03:33:53...
🔧 [TOOL EXECUTION] Querying mocked ERP database for production starting at: 04:13:53...
🔧 [TOOL EXECUTION] Scanning simulated Google Drive for shipping documents related to: LPN-260724-7153...



# OFFICIAL URGENT RECALL REPORT

**DATE:** August 22, 2026  
**STATUS:** CRITICAL / IMMEDIATE ACTION REQUIRED  
**INCIDENT TYPE:** Microbiological Contamination (*Escherichia coli*)  
**INVESTIGATOR:** Panthoron TraceAudit Agent (Senior Quality Manager)  

---

### 1. INCIDENT OVERVIEW
On August 22, 2026, an urgent notification was received from our supplier regarding a severe *Escherichia coli* (*E. coli*) contamination in **Raw Material Lot: 260707AH**. In accordance with IFS, BRC, and ISO food safety standards, an immediate traceability investigation was launched to identify, locate, and quarantine all affected products.

---

### 2. PRODUCTION TRACEABILITY & TIMELINE
The contaminated raw material was traced through our ERP database to determine its exact usage on the factory floor:

*   **Production Line:** Line 1
*   **Raw Material Drop Time:** 03:33:53
*   **Final Product Code:** 09118
*   **Line 1 Processing Parameters:**
    *   *Conveyor Transit Time:* 5 minutes
    *   *Freezing Tunnel Time:* 35 minutes
*   **Calculated Freezing Tunnel Exit Time:** **04:13:53** (Exact time the contaminated product exited the freezing tunnel and was packaged).

---

### 3. AFFECTED INVENTORY & PALLET IDENTIFICATION
Using the calculated exit window, the industrial ERP database was queried to identify the specific Master Pallet License Plate Number (LPN):

*   **Affected Pallet ID:** `LPN-260724-7153`
*   **Internal Database Status:** Successfully updated to **'BLOCKED'** to prevent any further internal movement or accidental dispatch.

---

### 4. LOGISTICS & SHIPPING STATUS (CRITICAL)
A scan of our logistics and shipping documentation (OCR/PDF analysis of Google Drive records) revealed that the affected pallet has already bypassed internal dispatch:

*   **Shipping Status:** **SHIPPED / IN TRANSIT**
*   **Associated Shipping Document:** `24072026FINAL.pdf`
*   **Consignee/Customer:** M. OGKOUNSOTO M.IKE
*   **Delivery Address:** Tsimiski 82, Thessaloniki
*   **Transport Vehicle License Plate:** **NBX7849**

---

### 5. IMMEDIATE CORRECTIVE & PREVENTIVE ACTIONS (CAPA)
To comply with international food safety regulations (IFS/BRC/ISO), the following actions must be executed immediately:

1.  **Customer Notification:** Contact **M. OGKOUNSOTO M.IKE** immediately to inform them of the contamination and instruct them to reject/quarantine **Pallet LPN-260724-7153** upon arrival.
2.  **Carrier Interception:** Contact the transport coordinator to locate vehicle **NBX7849** and instruct the driver to halt delivery and return the cargo to the facility under quarantine conditions, if feasible.
3.  **Internal Quarantine:** Ensure that the 'BLOCKED' status of `LPN-260724-7153` is strictly maintained in the ERP system to prevent any automated re-routing.
4.  **Supplier Audit & Root Cause Analysis:** Initiate a formal investigation into the supplier of Raw Material Lot: 260707AH and suspend further intake from this lot until corrective actions are verified.